# April experiment — NVDA, INTC & IBM, fully separated, original-main() defaults

Runs the experiment from `LearningKraus.py`'s original `main()` (as committed on the
**locked `main` baseline**, tag `baseline`) end to end, from raw data:

1. **Configs** — every April-2025 trading day on disk, one config per symbol,
   `instrument_filter: true` so NVDA, INTC, and IBM are **entirely separated** (own
   filtered event stream, own feature cache, own distributions, own models — they
   share nothing but the read-only raw files). NVDA and INTC read from
   `data/NVDA_INTC` (interleaved); IBM reads from its own `data/IBM`
   (`data.asset_paths` in `configs/default.yaml` maps each symbol to its directory).
2. **Distributions** — featurize → encode → `SEQ_DISTR_*`/`CLS_DISTR_*` per symbol.
3. **Training** — 3 symbols × 3 predictors = **9 models**, each a verbatim re-run of the
   original `main()` body. Per model you get the **2 result files + 4 charts**
   (9×2 = 18 files, 9×4 = 36 images), shown inline below as each model completes —
   forward each bundle as soon as it appears.

Every default in the next cell is the value from the baseline `main()` — change them
only to deviate from the original experiment. Training is unseeded (like the original)
unless you set `SEED`.

*Long runs over SSH: JupyterLab keeps the kernel alive if the browser disconnects; from a
plain terminal you can run the same thing detached with `scripts/run_april.py` under
`nohup`/`tmux`.*

In [ ]:
# ---- parameters, set for the LINUX COMPUTE ENVIRONMENT ----
# (training defaults = LearningKraus.main() on the locked baseline;
#  distribution/ensemble scope = colleague's email spec)
SYMBOLS = ["NVDA", "INTC", "IBM"]                 # per boss: all three, separated
PREDICTED = "log_mid"    # features[0] in ALL colleague files: every SEQ/CLS/ENS
                         # output pairs log_mid with predictor(s) — log_mid is
                         # the thing being predicted, never a "predictor" itself
PREDICTORS = ["tvi_n", "obi_L1", "ofi_L1_n_norm"]  # TRAINING: boss's 3 models/symbol

# distribution stage: colleague's full spec (his email / cls_reference.py) —
# 10 predictors (superset of the 3 above) x 5 classes, v2 (-1,0,1) CLS order
DIST_PREDICTORS = ["tvi_n", "obi_L1", "ofi_L1_n", "ofi_L1_n_norm",
                   "ofi_L1_norm_n", "ofi_L3_norm_n", "ofi_L10_norm_n",
                   "micro_price", "vpin", "sigma_W"]

EPOCHS = 3000        # main(): epochs=3000
N_QUBITS = 3         # main(): n_qubits = 3
SEED = None          # main() is unseeded; set an int for reproducible runs
# (batch_size=6*512, lr=1e-3, adam, nll_seq, learn_rho0=True, max_seq_len=6,
#  min_seq_prob=0.0, m=16, num_workers=8, device=cuda-else-cpu are fixed
#  inside the harness at exactly the original values; CUDA is picked up
#  automatically on the box)

# per-symbol raw-data directory: {} = use configs/default.yaml's
# data.asset_paths catalog as-is (NVDA/INTC -> data/NVDA_INTC, IBM -> data/IBM);
# override individual entries here, e.g. {"IBM": "/mnt/other/data/IBM"}
ASSET_PATH_OVERRIDES = {}
WORKERS = 0          # LINUX: one featurize worker per core  (Mac: use 4)
SKIP_DISTRIBUTIONS = False
RUN_ENSEMBLE = True  # LINUX: build the 20 v2 ENS_TD_* tables per symbol too


In [ ]:
import json, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import ASSET_CATALOG, find_data_dir, april_dates, detect_pattern, make_config
from pipeline.config import RunConfig

ASSET_CATALOG.update(ASSET_PATH_OVERRIDES)

# each symbol resolves its own directory (NVDA/INTC share NVDA_INTC; IBM is
# separate); the day-glob only runs once per distinct directory
resolved = {s: find_data_dir(s) for s in SYMBOLS}
for s, d in resolved.items():
    assert d.is_dir(), f"data dir not found for {s}: {d}"

configs = {}
scope_by_dir = {}
for s, data_dir in resolved.items():
    if data_dir not in scope_by_dir:
        pattern = detect_pattern(data_dir)
        dates = april_dates(data_dir, pattern)
        scope_by_dir[data_dir] = (pattern, dates)
        print(f"data: {data_dir}  (pattern: {pattern})")
        print(f"April days: {len(dates)} ({dates[0]}..{dates[-1]})")
    pattern, dates = scope_by_dir[data_dir]
    configs[s] = make_config(s, data_dir, dates, WORKERS, DIST_PREDICTORS, pattern,
                             epochs=EPOCHS, n_qubits=N_QUBITS,
                             seed=-1 if SEED is None else SEED,
                             train_predictors=PREDICTORS)

# every parameter of the run, in full — nothing is implicit
for s, cfg_path in configs.items():
    print(f"\n{'='*30} {s}: {cfg_path.name} {'='*30}")
    print(cfg_path.read_text())

### Stage 2 — per-symbol distributions
One decode+featurize per (symbol, day), day-parallel; outputs land in
`outputs/april/{SYMBOL}/`. Skipped if `SKIP_DISTRIBUTIONS = True`.

In [ ]:
if not SKIP_DISTRIBUTIONS:
    from pipeline.runner import run
    for symbol, cfg_path in configs.items():
        cfg = RunConfig.load(cfg_path)          # banner derives from the CONFIG
        print(f"=== distributions: {symbol} ({len(cfg.data.dates)} days x "
              f"{len(cfg.distributions.predictors)} predictors x "
              f"{len(cfg.distributions.class_names) or 1} classes) ===")
        t0 = time.time()
        run(cfg, run_id=f"april-{symbol}")
        print(f"{symbol} done in {time.time()-t0:.0f}s -> outputs/april/{symbol}/\n")
else:
    print("skipped (SKIP_DISTRIBUTIONS=True)")

### Stage 2b (optional) — ensemble training tables (`ENS_TD_*`)
The colleague's fixed-length multi-channel experiment, **v2**
(`ensemble_training_data_2.py`): 3 bivariate channels + 1 joint multivariate
channel (log_mid + ofi_L10_norm_n + micro_price + vpin, 256-symbol alphabet),
timestamp-aligned, 5 sequence lengths × 4 class definitions → 20 pickles per
symbol under `outputs/april/{SYMBOL}/ensemble/` (`..._ALL` names). Counting
math is his code verbatim (byte-equivalence: `tests/verify_ensemble_v2.py`);
featurize is served from the same per-symbol cache as stage 2, so this adds
roughly 30 min per symbol warm. Enable with `RUN_ENSEMBLE = True`.

In [ ]:
if RUN_ENSEMBLE:
    from pipeline.ensemble import run_ensemble
    for symbol, cfg_path in configs.items():
        cfg = RunConfig.load(cfg_path)          # banner derives from the CONFIG
        print(f"=== ensemble tables: {symbol} "
              f"({len(cfg.ensemble.predictors)} channels x "
              f"{len(cfg.ensemble.seq_lengths)} lengths x "
              f"{len(cfg.ensemble.class_names)} classes) ===")
        t0 = time.time()
        outputs = run_ensemble(cfg, run_id=f"april-ensemble-{symbol}")
        print(f"{symbol}: {len(outputs)} ENS_TD files in "
              f"{time.time()-t0:.0f}s -> outputs/april/{symbol}/ensemble/\n")
else:
    print("skipped (RUN_ENSEMBLE=False)")

### Stage 3 — the 9 models (send-as-you-go)
Each iteration is the original `main()` run for one (symbol, predictor). As each model
finishes, its **READY TO SEND** bundle is printed and the 4 charts render inline.
Progress persists in `outputs/april/april_summary.json` — safe to interrupt and re-run
with `SKIP_DISTRIBUTIONS = True`.

In [ ]:
from IPython.display import Image, display
from train_kraus_baseline import run_one

summary_path = ROOT / "outputs" / "april" / "april_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
results = []
combos = [(s, p) for s, cfg_path in configs.items()
          for p in RunConfig.load(cfg_path).training.predictors]

for i, (symbol, predictor) in enumerate(combos, 1):
    distr_dir = ROOT / "outputs" / "april" / symbol
    out_dir = distr_dir / "models"
    out_dir.mkdir(parents=True, exist_ok=True)
    tcfg = RunConfig.load(configs[symbol])
    t = tcfg.training
    print(f"\n=== MODEL {i}/{len(combos)}: {symbol} x {predictor} "
          f"(epochs={t.epochs}, {t.n_qubits}q, batch={t.batch_size}, "
          f"lr={t.lr}, {t.optimizer}/{t.loss_kind}, "
          f"seed={'unseeded' if t.seed < 0 else t.seed}) ===")
    r = run_one(predictor, distr_dir, out_dir, t.epochs,
                None if t.seed < 0 else t.seed,
                symbol=symbol, n_qubits=t.n_qubits, m=tcfg.alphabet_size,
                max_seq_len=t.max_seq_len, min_seq_prob=t.min_seq_prob,
                batch_size=t.batch_size, lr=t.lr,
                optimizer_name=t.optimizer, loss_kind=t.loss_kind,
                learn_rho0=t.learn_rho0, device=t.device)
    results.append(r)
    summary_path.write_text(json.dumps(results, indent=1))

    print(f"\n>>> MODEL {i}/{len(combos)} COMPLETE — READY TO SEND:")
    print(f"    result file 1: {r['model_pickle']}")
    print(f"    result file 2: {r['weights']}")
    for png in r["plots"]:
        print(f"    image:         {png}")
    print(f"    cost={r['final_cost_weighted_mse']:.3e}  ({r['train_seconds']:.0f}s)")
    for png in r["plots"]:
        display(Image(filename=png))

print(f"\nAll {len(combos)} models done. Summary: {summary_path}")

### Where everything lands
```
configs/april_nvda.yaml, april_intc.yaml, april_ibm.yaml   the experiment definitions
outputs/april/{SYMBOL}/SEQ_DISTR_*, CLS_*     per-symbol distributions
outputs/april/{SYMBOL}/feature_cache/         per-symbol featurized days
outputs/april/{SYMBOL}/ensemble/ENS_TD_*      20 v2 ensemble tables (stage 2b)
outputs/april/{SYMBOL}/models/MOD*, WGHTS_*   2 result files per model
outputs/april/{SYMBOL}/models/*_{n}q_1..4.png 4 charts per model
outputs/april/april_summary.json              cost + timing per model
```